In [2]:
import pandas as pd 
import numpy as np 


In [3]:
# load all data
train = pd.read_csv('../data/raw/train_values.csv')
labels = pd.read_csv('../data/raw/train_labels.csv')
test = pd.read_csv('../data/raw/test_values.csv')


In [4]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    floors = df["count_floors_pre_eq"].replace(0, np.nan)
    area = df["area_percentage"].replace(0, np.nan)

    # --- Existing features ---
    df["age_clipped"] = df["age"].clip(0, 100)
    df["age_bin_old"] = (df["age_clipped"] >= 30).astype(int)
    df["age_is_zero"] = (df["age"] == 0).astype(int)  # NEW: 10% of data

    df["height_per_floor"] = df["height_percentage"] / floors
    df["area_per_floor"] = df["area_percentage"] / floors
    df["slenderness"] = df["height_percentage"] / area
    df["volume_proxy"] = df["area_percentage"] * df["height_percentage"]
    df["is_tall"] = (df["count_floors_pre_eq"] >= 4).astype(int)

    df["old_mud_stone"] = df["age_bin_old"] * df["has_superstructure_mud_mortar_stone"]

    df["families_per_floor"] = df["count_families"] / floors
    df["families_per_area"] = df["count_families"] / area
    df["has_multiple_families"] = (df["count_families"] > 1).astype(int)

    df["vertical_risk"] = df["count_floors_pre_eq"] * df["height_percentage"]

    # --- NEW: Superstructure features ---
    df["fragile_score"] = (
        df["has_superstructure_adobe_mud"]
        + df["has_superstructure_mud_mortar_stone"]
        + df["has_superstructure_bamboo"]
    )
    df["strong_score"] = (
        df["has_superstructure_rc_engineered"]
        + df["has_superstructure_cement_mortar_brick"]
    )
    df["material_risk"] = df["fragile_score"] - df["strong_score"]
    df["rc_any"] = (
        (df["has_superstructure_rc_engineered"]
         + df["has_superstructure_rc_non_engineered"]) > 0
    ).astype(int)
    df["mud_dominant"] = (
        df["fragile_score"] > df["strong_score"]
    ).astype(int)
    df["age_x_fragile"] = df["age_clipped"] * df["fragile_score"]
    df["floors_x_height"] = df["count_floors_pre_eq"] * df["height_percentage"]

    # --- NEW: Geo interaction ---
    df["geo1_geo2"] = (
        df["geo_level_1_id"] * 10000 + df["geo_level_2_id"]
    )
    df["geo2_geo3"] = (
        df["geo_level_2_id"] * 100000 + df["geo_level_3_id"]
    )

    # --- NEW: Foundation + roof combo (label as int) ---
    df["foundation_roof"] = (
        df["foundation_type"].astype(str) + "_" + df["roof_type"].astype(str)
    ).astype("category").cat.codes

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

In [5]:
train = feature_engineering(train)
test = feature_engineering(test)


In [6]:
train

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,fragile_score,strong_score,material_risk,rc_any,mud_dominant,age_x_fragile,floors_x_height,geo1_geo2,geo2_geo3,foundation_roof
0,802906,6,487,12198,2,30,6,5,t,r,...,2,0,2,0,1,60,10,60487,48712198,6
1,28830,8,900,2812,2,10,8,7,o,r,...,1,0,1,0,1,10,14,80900,90002812,6
2,94947,21,363,8973,2,10,5,5,t,r,...,1,0,1,0,1,10,10,210363,36308973,6
3,590882,22,418,10694,2,10,6,5,t,r,...,2,0,2,0,1,20,10,220418,41810694,6
4,201944,11,131,1488,3,30,8,9,t,r,...,1,0,1,0,1,30,27,110131,13101488,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260596,688636,25,1335,1621,1,55,6,3,n,r,...,1,0,1,0,1,55,3,251335,133501621,6
260597,669485,17,715,2060,2,0,6,5,t,r,...,1,0,1,0,1,0,10,170715,71502060,6
260598,602512,17,51,8163,3,55,6,7,t,r,...,1,0,1,0,1,55,21,170051,5108163,7
260599,151409,26,39,1851,2,10,14,6,t,r,...,0,1,-1,0,0,0,12,260039,3901851,8


In [7]:
train

,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,fragile_score,strong_score,material_risk,rc_any,mud_dominant,age_x_fragile,floors_x_height,geo1_geo2,geo2_geo3,foundation_roof
0,802906,6,487,12198,2,30,6,5,t,r,...,2,0,2,0,1,60,10,60487,48712198,6
1,28830,8,900,2812,2,10,8,7,o,r,...,1,0,1,0,1,10,14,80900,90002812,6
2,94947,21,363,8973,2,10,5,5,t,r,...,1,0,1,0,1,10,10,210363,36308973,6
3,590882,22,418,10694,2,10,6,5,t,r,...,2,0,2,0,1,20,10,220418,41810694,6
4,201944,11,131,1488,3,30,8,9,t,r,...,1,0,1,0,1,30,27,110131,13101488,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260596,688636,25,1335,1621,1,55,6,3,n,r,...,1,0,1,0,1,55,3,251335,133501621,6
260597,669485,17,715,2060,2,0,6,5,t,r,...,1,0,1,0,1,0,10,170715,71502060,6
260598,602512,17,51,8163,3,55,6,7,t,r,...,1,0,1,0,1,55,21,170051,5108163,7
260599,151409,26,39,1851,2,10,14,6,t,r,...,0,1,-1,0,0,0,12,260039,3901851,8


In [8]:
train.isnull().sum().sort_values(ascending=False).head(20)

building_id                               0
geo_level_1_id                            0
geo_level_2_id                            0
geo_level_3_id                            0
count_floors_pre_eq                       0
age                                       0
area_percentage                           0
height_percentage                         0
land_surface_condition                    0
foundation_type                           0
roof_type                                 0
ground_floor_type                         0
other_floor_type                          0
position                                  0
plan_configuration                        0
has_superstructure_adobe_mud              0
has_superstructure_mud_mortar_stone       0
has_superstructure_stone_flag             0
has_superstructure_cement_mortar_stone    0
has_superstructure_mud_mortar_brick       0
dtype: int64

In [9]:
labels = pd.read_csv('../data/raw/train_labels.csv')


In [10]:
train= pd.merge(train, labels, on='building_id')

In [11]:
train.to_csv('../data/preprocessed/train.csv')
test.to_csv('../data/preprocessed/test.csv')


In [12]:
train_preprocessed = pd.read_csv('../data/preprocessed/train.csv')
test_preprocessed = pd.read_csv('../data/preprocessed/test.csv')



In [13]:
train_preprocessed['damage_grade']

0         3
1         2
2         3
3         2
4         3
         ..
260596    2
260597    3
260598    3
260599    2
260600    3
Name: damage_grade, Length: 260601, dtype: int64

In [14]:
train_preprocessed.isna().sum()

Unnamed: 0         0
building_id        0
geo_level_1_id     0
geo_level_2_id     0
geo_level_3_id     0
                  ..
floors_x_height    0
geo1_geo2          0
geo2_geo3          0
foundation_roof    0
damage_grade       0
Length: 64, dtype: int64

In [15]:
print(train['count_floors_pre_eq'].min())
print(train['area_percentage'].min())


1
1


In [16]:
train.groupby('land_surface_condition')['damage_grade'].mean().sort_values()


land_surface_condition
t    2.234170
n    2.251407
o    2.289081
Name: damage_grade, dtype: float64